# 13. A generated campaign: the penicillin deep win, put to a proper test

**Answers to the questions fixed in advance** (details in the sections below; a win needs 12 points with a range clear of zero):

- **Q1. Yes.** With 30 batches of the published fault (A), the autoencoder catches 0.98 of faulty readings, 17 points more than the best of the previous simple detectors (the curved-link scored like the autoencoder, 0.81): a win by the margin. But its line carries over less well: on the faulty batches' own normal readings it raises false alarms 9% of the time, where the line is set for 1%.
- **Q2. Yes, and more.** The unseen value check catches every reading of fault A (1.00), as does the nearest normal reading: the autoencoder is 2 points behind them.
- **Q3. Yes.** Held at values the recipe does use (fault B), the unseen value check misses it entirely (0.00), as designed. The nearest normal reading still catches every reading (1.00), the autoencoder 0.90; recipe at this hour only 0.15.
- **Q4.** Only the schedule-aware checks and the deep models see fault C. Each catches about 0.8 at 80% of schedule (the smallest fault) and 0.94 to 1.00 at 50% and 20%. The link detectors see only the 20% fault (0.80); per-sensor checks, PCA and the shape-in-time detectors see almost nothing at any size.
- **Q5. The schedule-aware checks keep their edge; the autoencoder does not.** With operator-run batches in the normal data, the unseen value check still catches all of fault A (though on operator batches it raises 11% false alarms), while the autoencoder falls from 0.98 to 0.78 (recipe batches) and 0.72 (operator batches).
- **Q6. No.** On the aeration, back pressure, base flow and coolant flow faults no deep model beats the best simple detector: the first two are caught whole by simple detectors and deep models alike, and on base flow a simple link detector leads.
- **Overall: no deep model beats the best simple detector on any generated fault.** Against the detectors this project started with, the autoencoder's win on the substrate feed fault holds with 30 batches; against simple checks that know the schedule, it does not.

**Why.** The autoencoder's one win (notebook 09) rested on 10 faulty penicillin batches, three of which carried all of it, and round 3 found that a simple check that knows which values the recipe uses matches it (notebook 09, section 12). This notebook generates a campaign of 320 batches with PenSimPy, the Python port of the simulator (with the fixes in `adlab/pensim.py`), and asks six questions fixed in advance.

**The batches** (`adlab.pensim.campaign_plan`; every seed recorded in `data/raw/pensim_campaign/plan.json`, not committed). Lengths vary as in the published set: round(230 + 25 × a normal draw) hours, 148 to 290.

| Group | Batches | What |
| --- | --- | --- |
| Normal, recipe-driven | 100 | |
| Normal, operator-run | 60 | random steps on the sugar and precursor feeds (PenSimPy's `PRBS = 1`) |
| Fault A, recipe-driven | 30 | the substrate feed fault as published: sugar feed held at 2 L/h for 20-30 h, at 20 L/h for 76-92 h and 200-214 h |
| Fault B, recipe-driven | 30 | the same hours, held at values the recipe uses at other times, fixed before generating: 8 L/h, then 15 L/h |
| Fault C, recipe-driven | 15 + 15 + 15 | the sugar feed at 80%, 50% and 20% of its scheduled value over the same hours |
| Fault A, operator-run | 15 | |
| Aeration, back pressure, base flow, coolant flow | 10 each | the simulator's own faults, at published settings, recipe-driven |

**Splits, fixed before computing.** Recipe normal: the first 60 to fit (every fifth of them, 12, held aside for the deep models' early stopping), the other 40 to set alarm lines. Operator normal: 40 to fit, 20 for lines. Two setups: the **recipe setup** (fit and lines on recipe batches only; scores every recipe-driven fault) and the **mixed setup** (fit and lines on recipe and operator batches together; scores fault A on both kinds of batch).

**Detectors:** the eight standard ones, the kinked-link, the straight forecaster, PCA and the curved-link scored like the autoencoder, the four schedule-aware checks (notebook 09, section 12), the autoencoder (all channels, and the 11 controller outputs only) and the LSTM forecaster, all with notebook 09's IndPenSim settings and no retuning (the LSTM at IndPenSim's choice, window 5 and two layers of 64; the straight forecaster at window 10; the autoencoder's bottleneck by its rule), three seeds where a detector has seeds.

**Questions, fixed before computing.** Q1. With 30 batches, does the autoencoder beat the best of the previous simple detectors on fault A by the margin? Q2. Does any schedule-aware check come within 12 points of the autoencoder on fault A? Q3. Fault B: the unseen value check should miss it by design; do the nearest normal reading, recipe at this hour and the autoencoder still catch it? Q4. Fault C: catch rate against fault size, per detector family. Q5. Mixed setup: with the operators' random steps in normal data, do the schedule-aware checks lose their edge, and does the autoencoder keep its? Q6. The other four faults: does any deep model beat the best simple detector? Ranges from resampling batches; a win needs 12 points with a range clear of zero.

**Two things found while making the batches** (before any scoring): PenSimPy's operator mode never applied its random steps (it read a step from the wrong place in a list; fixed in `adlab/pensim.py`, and the 75 operator-run batches were made again), and its batch length can vary with a small change (also in `adlab/pensim.py`).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
import pandas as pd

from adlab import campaign, explore

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", explore.tidy)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
plan = campaign.plan()
plan.groupby("group").agg(batches=("seed", "size"), first_seed=("seed", "min"), last_seed=("seed", "max"),
                          shortest_hours=("hours", "min"), longest_hours=("hours", "max"))

,batches,first_seed,last_seed,shortest_hours,longest_hours
group,,,,,
A_operator,15,3500,3514,148,269
A_recipe,30,3000,3029,186,265
B_recipe,30,3100,3129,183,272
C20_recipe,15,3400,3414,207,272
C50_recipe,15,3300,3314,163,265
C80_recipe,15,3200,3214,191,263
aeration_recipe,10,4100,4109,192,290
base_flow_recipe,10,4400,4409,202,281
coolant_recipe,10,4500,4509,188,283


## 1. The generated batches against the published ones

Each channel's 1st percentile, median and 99th percentile in the normal batches, generated against published, for the recipe-driven and the operator-run batches; then the curved share (notebook 08's measure) on 60 generated recipe batches and on the 30 published recipe batches.

In [2]:
for strategy in ("recipe", "operator"):
    r = campaign.compare_ranges(strategy)
    print(strategy, "| batches:", r.attrs["batches"], "| hours:", r.attrs["batch hours"])
    display(r)
campaign.curved_share()

recipe | batches: {'generated': 100, 'published': 30} | hours: {'generated': (np.float64(180.99999999999983), np.float64(289.99999999999983)), 'published': (179.0, 290.0)}


,generated: 1%,generated: median,generated: 99%,published: 1%,published: median,published: 99%
channel,,,,,,
dissolved oxygen,9.479,13.226,16.055,8.825,12.501,15.276
volume,57750.305,85159.676,98017.909,57440,75568,85979.28
weight,61883.954,93956.986,107529.836,61667.44,84084,95071.76
pH,6.264,6.5,6.59,6.394,6.501,6.6
temperature,297.844,297.973,298.967,297.85,297.99,298.558
CO2 off-gas,0.124,1.63,2.103,0.124,1.588,2.098
oxygen uptake,0.02,1.287,1.96,0.022,1.277,2.149
O2 off-gas,0.183,0.191,0.205,0.182,0.191,0.205
CO2 evolution,0.046,1.422,1.885,0.047,1.403,1.881


operator | batches: {'generated': 60, 'published': 30} | hours: {'generated': (np.float64(183.99999999999983), np.float64(281.99999999999983)), 'published': (193.0, 262.0)}


,generated: 1%,generated: median,generated: 99%,published: 1%,published: median,published: 99%
channel,,,,,,
dissolved oxygen,9.45,13.346,16.359,7.515,12.765,15.588
volume,57796.124,85144.044,99085.738,57517,76192,86296.04
weight,61961.128,93887.163,108932.263,61733,84732,95682.12
pH,6.062,6.5,6.601,6.422,6.501,6.625
temperature,297.841,297.973,298.897,297.83,297.99,298.69
CO2 off-gas,0.126,1.603,2.13,0.12,1.609,2.087
oxygen uptake,0.028,1.29,1.964,0.006,1.27,2.095
O2 off-gas,0.183,0.191,0.204,0.182,0.191,0.205
CO2 evolution,0.046,1.411,1.877,0.046,1.409,1.89


,straight share,"curved share, points",range
batches,,,
generated recipe batches (60),0.713,16.024,+15.0 to +16.9
published recipe batches (30),0.57,20.639,+15.6 to +33.3


**What this shows. The generated batches are close to the published ones, with the same differences as the pilot of part C.**

- **Lengths now match:** generated recipe batches run 181 to 290 hours (published 179 to 290), operator-run ones 184 to 282 (published 193 to 262).
- **The recipe's controller outputs are the same** (air flow, feeds, pressure, water, precursor, discharge), and so are the measurements' ranges (dissolved oxygen, pH, temperature, off-gas, oxygen uptake, CO2 evolution).
- **Where they differ:** the volume and weight run about 10% higher (median 85,000 against 76,000 L); the base flow about twice as high (median 113 against 55 L/h); the cooling water reaches higher peaks (99th percentile 833 against 437 L/h); and the operator-run batches' pH dips lower (1st percentile 6.06 against 6.42).
- **The operators' random steps now match:** the generated operator batches add the same sugar feed values the published operators use and the recipe does not (45, 60, 105, 120 and 135 L/h; plus a rare 0 and 165), and move the precursor flow between 3 and 14 (published 5 to 15).
- **The curved share is in the same range:** +16.0 points (+15.0 to +16.9) on 60 generated recipe batches, against +20.6 (+15.6 to +33.3) on the 30 published recipe batches.

## 2. Scoring

Every detector, both setups (cached; about an hour the first time).

In [3]:
for setup in ("recipe", "mixed"):
    campaign.score(setup)          # about an hour the first time; then read from results/cache

## 3. Q1, Q2 and Q6: every fault in the recipe setup

For each fault: the autoencoder's catch rate; the best of the previous simple detectors (everything but the schedule-aware checks) and the autoencoder's gap to it (Q1); the best schedule-aware check and the gap (Q2); the best simple detector of all; and the best deep model against the best simple detector (Q6).

In [4]:
campaign.questions("recipe")

,faulty batches,autoencoder,best previous simple,"autoencoder minus best previous simple, points","autoencoder minus best previous simple, range","autoencoder minus best previous simple, verdict",best schedule-aware check,"autoencoder minus best schedule-aware check, points","autoencoder minus best schedule-aware check, range","autoencoder minus best schedule-aware check, verdict",best simple of all,"autoencoder minus best simple of all, points","autoencoder minus best simple of all, range","autoencoder minus best simple of all, verdict",best deep,"best deep minus best simple, points","best deep minus best simple, range","best deep minus best simple, verdict"
fault (recipe setup),,,,,,,,,,,,,,,,,,
A: as published,30,0.981,"Curved-link, scored like the autoencoder (added) (0.81)",17.284,+3.5 to +95.3,ahead,Unseen value check (1.00),-1.934,-7.5 to -1.9,tie,Unseen value check (1.00),-1.934,-7.5 to -1.9,tie,"Autoencoder, outputs only (0.99)",-1.401,-4.8 to +1.2,tie
B: held at recipe values,30,0.9,"Curved-link, scored like the autoencoder (added) (0.90)",0.077,-7.8 to +74.6,tie,Nearest normal reading (k = 1) (1.00),-9.964,-31.8 to -9.0,tie,Nearest normal reading (k = 1) (1.00),-9.964,-31.8 to -9.0,tie,"Autoencoder, outputs only (0.95)",-4.75,-10.1 to +3.9,tie
C: 80% of schedule,15,0.748,Matrix Profile (0.06),69.099,+41.9 to +72.1,ahead,Nearest normal reading (k = 1) (0.77),-1.883,-33.7 to +3.3,tie,Nearest normal reading (k = 1) (0.77),-1.883,-33.7 to +3.3,tie,"Autoencoder, outputs only (0.80)",3.676,+1.9 to +4.8,tie
C: 50% of schedule,15,0.878,Matrix Profile (0.03),84.612,+73.4 to +88.3,ahead,Nearest normal reading (k = 1) (1.00),-12.244,-23.4 to -4.2,behind,Nearest normal reading (k = 1) (1.00),-12.244,-23.4 to -4.2,behind,"Autoencoder, outputs only (0.96)",-4.481,-7.8 to +0.4,tie
C: 20% of schedule,15,0.858,Curved-link residual (0.80),5.382,-4.1 to +73.4,tie,Nearest normal reading (k = 1) (1.00),-14.086,-21.0 to -13.9,behind,Nearest normal reading (k = 1) (1.00),-14.086,-21.0 to -13.9,behind,"Autoencoder, outputs only (0.94)",-5.969,-13.4 to -0.6,tie
aeration,10,1,PCA (T2 + SPE) (1.00),0,+0.0 to +0.0,tie,Unseen value check (1.00),0,+0.0 to +0.0,tie,PCA (T2 + SPE) (1.00),0,+0.0 to +0.0,tie,Autoencoder (1.00),0,+0.0 to +0.0,tie
back pressure,10,1,PCA (T2 + SPE) (1.00),0,+0.0 to +5.9,tie,Unseen value check (1.00),0,+0.0 to +0.0,tie,PCA (T2 + SPE) (1.00),0,+0.0 to +5.9,tie,Autoencoder (1.00),0,+0.0 to +5.9,tie
base flow,10,0.625,"Curved-link, scored like the autoencoder (added) (0.92)",-29.836,-36.4 to +11.3,tie,Nearest normal readings (k = 5) (0.04),59.016,+17.9 to +72.8,ahead,"Curved-link, scored like the autoencoder (added) (0.92)",-29.836,-36.4 to +11.3,tie,LSTM forecaster (0.69),-23.361,-26.6 to +51.9,tie
coolant flow,10,0.482,Curved-link residual (0.44),4.137,+2.3 to +8.3,tie,Nearest normal reading (k = 1) (0.02),46.593,+35.9 to +61.7,ahead,Curved-link residual (0.44),4.137,+2.3 to +8.3,tie,Autoencoder (0.48),4.137,+2.3 to +8.3,tie


**What this shows.** (Ranges in the first table come from resampling batches, for the gap between two detectors; the 12-point margin applies.)

- **Q1. Yes: with 30 batches, the autoencoder beats the best of the previous simple detectors on fault A by the margin.** It catches **0.98** of fault A's readings, against 0.81 for the best previous simple detector (the curved-link scored like the autoencoder): +17 points (+3.5 to +95). Its line carries over less well than theirs: on the faulty batches' own normal readings it raises false alarms 9% of the time (the curved-link scored alike about 5%, the unseen value check none). Nearly every reading of the fault is caught this time, not only the 76-92 h stretch as on the published batches. The likely reason (not tested here): fitted on recipe batches alone, the autoencoder learns the recipe's islands more tightly than on the published set, whose fitting batches mixed three ways of running the plant.
- **Q2. Yes, and more: a schedule-aware check catches more than the autoencoder.** The unseen value check catches **every** reading of fault A (1.00; the autoencoder is 1.9 points behind it, −7.5 to −1.9), with no false alarms on the faulty batches' own normal readings. The nearest normal reading catches 1.00 too.
- **Q6. No deep model beats the best simple detector on the other four faults.** The aeration and back pressure faults are caught whole by PCA, the unseen value check and the autoencoder alike (ties at 1.00). On the base flow fault the curved-link scored like the autoencoder (0.92) leads the best deep model (the LSTM, 0.69); on the coolant flow fault the autoencoder (0.48) and the curved-link residual (0.44) tie.
- **Across every fault of this setup, the best deep model against the best simple detector is a tie** (−23 to +4 points, none by the margin).

## 4. Q3: fault B, held at values the recipe does use

In [5]:
campaign.q3_table()

,family,caught (mean of seeds),range low (seed 0),range high (seed 0),false alarms on own normal readings
method,,,,,
Nearest normal reading (k = 1),schedule-aware checks,1,1,1,0.042
Nearest normal readings (k = 5),schedule-aware checks,1,1,1,0.041
"Autoencoder, outputs only",deep,0.952,0.804,0.944,0.03
Autoencoder,deep,0.9,0.77,0.998,0.091
Recipe at this hour,schedule-aware checks,0.145,0.037,0.261,0.035
LSTM forecaster,deep,0.014,0.002,0.005,0.012
Unseen value check,schedule-aware checks,0,0,0,0


**What this shows. Q3: yes, the nearest normal reading and the autoencoder still catch fault B; the unseen value check misses it, as designed.** Held at 8 and then 15 L/h, values the recipe does use early in the batch, fault B gives the unseen value check nothing to see (0.00). The **nearest normal reading** catches **every** reading (1.00, k = 1 and k = 5): those values are normal on their own but never with the other outputs of those hours. The **autoencoder** catches **0.90** (0.95 on the controller outputs alone). **Recipe at this hour** catches only 0.15: its line is still set by the noisy cooling-water and base-flow controllers (notebook 09, section 12). The LSTM catches almost nothing (0.01); likely because a steady held value soon shows in its recent readings, from which it forecasts (not tested here).

## 5. Q4: catch rate against fault size

Each family's best detector at each size of fault C, with fault A for reference. Families: the per-sensor checks (rolling z, EWMA, CUSUM), PCA (T² + SPE and scored like the autoencoder), the links (straight, kinked, curved, curved scored alike), shape in time (matrix profile, straight forecaster, isolation forest), the schedule-aware checks and the deep models.

In [6]:
campaign.q4_table()

,per-sensor checks,PCA,links,shape in time,schedule-aware checks,deep,best detector
fault size,,,,,,,
C: 80% of schedule,0.015,0.014,0.014,0.057,0.767,0.804,"Autoencoder, outputs only"
C: 50% of schedule,0.014,0.008,0.02,0.031,1,0.955,Nearest normal reading (k = 1)
C: 20% of schedule,0.027,0.017,0.805,0.038,0.999,0.94,Nearest normal reading (k = 1)
A: as published,0.022,0.015,0.808,0.02,1,0.986,Unseen value check


**What this shows. Q4: only the schedule-aware checks and the deep models see fault C at any size; the smallest fault is the hardest, and the two families track each other.**

| Sugar feed at | Per-sensor checks | PCA | Links | Shape in time | Schedule-aware checks | Deep models |
| --- | --- | --- | --- | --- | --- | --- |
| 80% of schedule | 0.02 | 0.01 | 0.01 | 0.06 | 0.77 | 0.80 |
| 50% of schedule | 0.01 | 0.01 | 0.02 | 0.03 | 1.00 | 0.96 |
| 20% of schedule | 0.03 | 0.02 | 0.80 | 0.04 | 1.00 | 0.94 |
| (A, as published) | 0.02 | 0.01 | 0.81 | 0.02 | 1.00 | 0.99 |

(Each family's best detector.) A sugar feed at 80% of schedule is a small shift, and it is the one fault where a deep model (the autoencoder on the controller outputs, 0.80) edges the best schedule-aware check (the nearest normal reading, 0.77): +3.7 points (+1.9 to +4.8), far under the margin. At 50% and 20%, the nearest normal reading catches everything. The link detectors see only the largest shift (20%, and fault A). The per-sensor checks, PCA and the shape-in-time detectors see almost nothing at any size: a lower feed rate is still inside every sensor's normal range.

## 6. Q5: the mixed setup, with the operators' random steps in normal data

In [7]:
campaign.q5_table()

caught  \
setup and fault                    detector                                 
recipe setup, A: as published      Unseen value check                   1   
                                   Nearest normal reading (k = 1)       1   
                                   Nearest normal readings (k = 5)      1   
                                   Recipe at this hour              0.146   
                                   Autoencoder                      0.981   
                                   Autoencoder, outputs only        0.986   
mixed setup, A: as published       Unseen value check                   1   
                                   Nearest normal reading (k = 1)       1   
                                   Nearest normal readings (k = 5)  0.934   
                                   Recipe at this hour              0.042   
                                   Autoencoder                      0.775   
                                   Autoencoder, outputs only        0.685   
mixed setup, A on operator batches Unseen value check                   1   
                                   Nearest normal reading (k = 1)    0.78   
                                   Nearest normal readings (k = 5)  0.717   
                                   Recipe at this hour              0.035   
                                   Autoencoder                      0.717   
                                   Autoencoder, outputs only         0.59   

                                                                        range  \
setup and fault                    detector                                     
recipe setup, A: as published      Unseen value check               1.00-1.00   
                                   Nearest normal reading (k = 1)   1.00-1.00   
                                   Nearest normal readings (k = 5)  1.00-1.00   
                                   Recipe at this hour              0.04-0.26   
                                   Autoencoder                      0.94-1.00   
                                   Autoencoder, outputs only        0.92-0.98   
mixed setup, A: as published       Unseen value check               1.00-1.00   
                                   Nearest normal reading (k = 1)   0.84-1.00   
                                   Nearest normal readings (k = 5)  0.68-1.00   
                                   Recipe at this hour              0.03-0.15   
                                   Autoencoder                      0.61-0.89   
                                   Autoencoder, outputs only        0.51-0.55   
mixed setup, A on operator batches Unseen value check               1.00-1.00   
                                   Nearest normal reading (k = 1)   0.57-0.85   
                                   Nearest normal readings (k = 5)  0.54-0.84   
                                   Recipe at this hour              0.03-0.14   
                                   Autoencoder                      0.63-0.80   
                                   Autoencoder, outputs only        0.47-0.52   

                                                                   false alarms on own normal readings  \
setup and fault                    detector                                                              
recipe setup, A: as published      Unseen value check                                                0   
                                   Nearest normal reading (k = 1)                                0.034   
                                   Nearest normal readings (k = 5)                               0.035   
                                   Recipe at this hour                                           0.034   
                                   Autoencoder                                                   0.087   
                                   Autoencoder, outputs only                                     0.029   
mixed setup, A: as published       Unseen value check    

**What this shows. Q5: the schedule-aware checks keep their edge in the mixed setup; the autoencoder does not keep its level.**

- **Fault A on recipe batches**, with operator-run batches now among the fitting and line batches: the unseen value check still catches **every** faulty reading (1.00, with no false alarms on the faulty batches' own normal readings), and the nearest normal reading 1.00. The autoencoder falls from 0.98 (recipe setup) to **0.78**, and on the controller outputs alone from 0.99 to 0.69: likely because the operators' random steps loosen the normal combinations of controller outputs that it learns (not tested here). It is 22 points behind the unseen value check, behind by the margin.
- **Fault A on operator-run batches:** the unseen value check catches 1.00 again, but here its line does not carry over: on these batches' own normal readings it raises false alarms **11% of the time**, eleven times the rate its line is set for, because an operator's random step now and then gives a value no fitting batch had. So that 1.00 is partly flattered. The nearest normal reading catches 0.78 (false alarms 3%), the autoencoder **0.72** (false alarms about 10%), a tie with the nearest normal reading.
- **Against the earlier simple detectors, the autoencoder's lead grows in this setup:** they collapse when the operators' steps enter the normal data (the best of them catches 0.02 of fault A on recipe batches and 0.04 on operator batches), so the autoencoder leads them by +75 and +68 points. Only the schedule-aware checks keep up, and do better.
- So in normal data with random steps, a lookup of seen values gets noisier but still catches the fault; the autoencoder loses about a fifth of its catch, and everything else loses almost all of it.

## 7. Every detector on every fault (recipe setup)

For reference: caught, its range, and false alarms on the faulty batches' own normal readings.

In [8]:
for v in campaign.SCORED["recipe"]:
    print(campaign.VARIANT_TITLES[v])
    display(campaign.variant_table("recipe", v))

A: as published


,family,caught (mean of seeds),range low (seed 0),range high (seed 0),false alarms on own normal readings
method,,,,,
Unseen value check,schedule-aware checks,1,1,1,0
Nearest normal readings (k = 5),schedule-aware checks,1,1,1,0.035
Nearest normal reading (k = 1),schedule-aware checks,1,1,1,0.034
"Autoencoder, outputs only",deep,0.986,0.924,0.984,0.029
Autoencoder,deep,0.981,0.944,1,0.087
"Curved-link, scored like the autoencoder (added)",links,0.808,0.001,0.945,0.052
Curved-link residual,links,0.758,0,0.958,0.047
Kinked-link residual,links,0.222,0.001,0.648,0.057
Recipe at this hour,schedule-aware checks,0.146,0.042,0.256,0.034


B: held at recipe values


,family,caught (mean of seeds),range low (seed 0),range high (seed 0),false alarms on own normal readings
method,,,,,
Nearest normal reading (k = 1),schedule-aware checks,1,1,1,0.042
Nearest normal readings (k = 5),schedule-aware checks,1,1,1,0.041
"Autoencoder, outputs only",deep,0.952,0.804,0.944,0.03
Autoencoder,deep,0.9,0.77,0.998,0.091
"Curved-link, scored like the autoencoder (added)",links,0.9,0.002,0.986,0.056
Curved-link residual,links,0.872,0,0.99,0.052
Kinked-link residual,links,0.157,0.002,0.624,0.073
Recipe at this hour,schedule-aware checks,0.145,0.037,0.261,0.035
Straight forecaster,shape in time,0.021,0.02,0.022,0.009


C: 80% of schedule


,family,caught (mean of seeds),range low (seed 0),range high (seed 0),false alarms on own normal readings
method,,,,,
"Autoencoder, outputs only",deep,0.804,0.76,0.787,0.014
Nearest normal reading (k = 1),schedule-aware checks,0.767,0.749,0.79,0.009
Autoencoder,deep,0.748,0.408,0.783,0.01
Unseen value check,schedule-aware checks,0.692,0.677,0.7,0
Nearest normal readings (k = 5),schedule-aware checks,0.576,0.117,0.778,0.008
Matrix Profile,shape in time,0.057,0.002,0.108,0.009
Recipe at this hour,schedule-aware checks,0.03,0.019,0.042,0.012
Straight forecaster,shape in time,0.02,0.02,0.021,0.007
Rolling z-score,per-sensor checks,0.015,0,0.034,0.01


... (8 outputs left out)

base flow


,family,caught (mean of seeds),range low (seed 0),range high (seed 0),false alarms on own normal readings
method,,,,,
"Curved-link, scored like the autoencoder (added)",links,0.924,0.071,0.929,0.066
Curved-link residual,links,0.919,0.041,0.925,0.044
PCA (T2 + SPE),PCA,0.732,0.174,0.863,0.041
LSTM forecaster,deep,0.69,0.123,0.277,0.024
"PCA, scored like the autoencoder",PCA,0.643,0,0.861,0.017
Autoencoder,deep,0.625,0.097,0.657,0.052
EWMA chart,per-sensor checks,0.554,0.005,0.768,0.066
Rolling z-score,per-sensor checks,0.525,0.01,0.75,0.062
Kinked-link residual,links,0.451,0,0.662,0.043


coolant flow


,family,caught (mean of seeds),range low (seed 0),range high (seed 0),false alarms on own normal readings
method,,,,,
Autoencoder,deep,0.482,0.369,0.651,0.019
Curved-link residual,links,0.441,0.33,0.596,0.022
"Curved-link, scored like the autoencoder (added)",links,0.434,0.322,0.579,0.025
"PCA, scored like the autoencoder",PCA,0.411,0.291,0.559,0.021
Straight-link residual,links,0.398,0.277,0.548,0.019
PCA (T2 + SPE),PCA,0.389,0.269,0.538,0.018
Kinked-link residual,links,0.378,0.259,0.535,0.018
LSTM forecaster,deep,0.35,0.259,0.508,0.016
Rolling z-score,per-sensor checks,0.304,0.168,0.452,0.024


## 8. Round 4: classic simple detectors and false alarms before the fault

*Added in round 4 (notebook 09, sections 14 and 15, has the design).* The classic simple detectors on both setups, the best simple detector against the best deep model again, and the false alarms each detector raises on the faulty batches' own readings before the fault.

In [9]:
from adlab import round4
display(round4.campaign_questions())
for setup in ("recipe", "mixed"):
    display(round4.campaign_classic(setup))

best simple, round 3  \
setup  fault                                                                               
recipe A: as published                                         Unseen value check (1.00)   
       B: held at recipe values                    Nearest normal reading (k = 1) (1.00)   
       C: 80% of schedule                          Nearest normal reading (k = 1) (0.77)   
       C: 50% of schedule                          Nearest normal reading (k = 1) (1.00)   
       C: 20% of schedule                          Nearest normal reading (k = 1) (1.00)   
       aeration                                                    PCA (T2 + SPE) (1.00)   
       back pressure                                               PCA (T2 + SPE) (1.00)   
       base flow                 Curved-link, scored like the autoencoder (added) (0.92)   
       coolant flow                                          Curved-link residual (0.44)   
mixed  A: as published                                         Unseen value check (1.00)   
       A on operator batches                                   Unseen value check (1.00)   

                                best deep minus best simple, round 3  \
setup  fault                                                           
recipe A: as published                       -1.4 (-4.8 to +1.2) tie   
       B: held at recipe values             -4.8 (-10.1 to +3.9) tie   
       C: 80% of schedule                    +3.7 (+1.9 to +4.8) tie   
       C: 50% of schedule                    -4.5 (-7.8 to +0.4) tie   
       C: 20% of schedule                   -6.0 (-13.4 to -0.6) tie   
       aeration                              +0.0 (+0.0 to +0.0) tie   
       back pressure                         +0.0 (+0.0 to +5.9) tie   
       base flow                          -23.4 (-26.6 to +51.9) tie   
       coolant flow                          +4.1 (+2.3 to +8.3) tie   
mixed  A: as published                 -22.5 (-40.0 to -12.0) behind   
       A on operator batches           -28.3 (-40.1 to -22.5) behind   

                                                                        best simple, round 4  \
setup  fault                                                                                   
recipe A: as published                                             Unseen value check (1.00)   
       B: held at recipe values                        Nearest normal reading (k = 1) (1.00)   
       C: 80% of schedule        Recipe at this hour, stepped outputs (fixed after seeing...   
       C: 50% of schedule                              Nearest normal reading (k = 1) (1.00)   
       C: 20% of schedule        Recipe at this hour, stepped outputs (fixed after seeing...   
       aeration                                                        PCA (T2 + SPE) (1.00)   
       back pressure                                                   PCA (T2 + SPE) (1.00)   
       base flow                          Nearest normal reading, all sensors (k = 1) (0.95)   
       coolant flow                       Nearest normal reading, all sensors (k = 1) (0.44)   
mixed  A: as published                                             Unseen value check (1.00)   
       A on operator batches                                       Unseen value check (1.00)   

                                best deep minus best simple, round 4  
setup  fault                                                          
recipe A: as published                       -1.4 (-4.8 to +1.2) tie  
       B: held at recipe values             -4.8 (-10.1 to +3.9) tie  
       C: 80% of schedule              -19.6 (-21.0 to -18.3) behind  
       C: 50% of schedule                    -4.5 (-7.8 to +0.4) tie  
       C: 20% of schedule                   -6.0 (-13.5 to -0.7) tie  
       aeration                              +0.0 (+0.0 to +0.0) tie  
       back pressure                         +0.0 (+0.0 to +5.9) tie  
       base flow                       -25.6 (

,A: as published,B: held at recipe values,C: 80% of schedule,C: 50% of schedule,C: 20% of schedule,aeration,back pressure,base flow,coolant flow
detector,,,,,,,,,
"Nearest normal reading, all sensors (k = 1)",1.00 (0.67-1.00),0.97 (0.72-1.00),0.03 (0.00-0.14),0.77 (0.01-0.90),0.88 (0.74-0.95),1.00 (1.00-1.00),1.00 (1.00-1.00),0.95 (0.88-0.97),0.44 (0.34-0.60)
"Nearest normal readings, all sensors (k = 5)",0.99 (0.36-1.00),0.90 (0.48-1.00),0.02 (0.00-0.11),0.51 (0.00-0.87),0.82 (0.42-0.92),1.00 (1.00-1.00),1.00 (1.00-1.00),0.94 (0.85-0.96),0.44 (0.33-0.58)
"Unseen value check, all sensors",1.00 (1.00-1.00),0.00 (0.00-0.00),0.69 (0.68-0.70),0.79 (0.78-0.80),0.90 (0.90-0.90),1.00 (1.00-1.00),1.00 (1.00-1.00),0.00 (0.00-0.00),0.00 (0.00-0.00)
Dynamic PCA (2 lags),0.02 (0.00-0.04),0.02 (0.00-0.03),0.02 (0.00-0.04),0.01 (0.00-0.02),0.02 (0.00-0.04),0.98 (0.94-1.00),1.00 (0.03-1.00),0.29 (0.00-0.55),0.33 (0.21-0.48)
CVA (T2 + Q),0.01 (0.00-0.04),0.01 (0.00-0.04),0.01 (0.00-0.02),0.00 (0.00-0.01),0.01 (0.00-0.03),1.00 (1.00-1.00),1.00 (1.00-1.00),0.20 (0.00-0.34),0.33 (0.18-0.47)
Kernel PCA (RBF),0.02 (0.00-0.12),0.02 (0.00-0.22),0.01 (0.00-0.03),0.00 (0.00-0.02),0.02 (0.00-0.12),1.00 (0.98-1.00),1.00 (1.00-1.00),0.67 (0.00-0.82),0.37 (0.23-0.53)
"Recipe at this hour, stepped outputs (fixed after seeing results)",1.00 (1.00-1.00),1.00 (1.00-1.00),1.00 (1.00-1.00),1.00 (1.00-1.00),1.00 (1.00-1.00),1.00 (1.00-1.00),1.00 (1.00-1.00),0.00 (0.00-0.00),0.00 (0.00-0.00)


,A: as published,A on operator batches
detector,,
"Nearest normal reading, all sensors (k = 1)",0.74 (0.54-0.99),0.68 (0.51-0.83)
"Nearest normal readings, all sensors (k = 5)",0.63 (0.44-0.88),0.60 (0.40-0.79)
"Unseen value check, all sensors",1.00 (1.00-1.00),1.00 (1.00-1.00)
Dynamic PCA (2 lags),0.01 (0.01-0.02),0.02 (0.01-0.03)
CVA (T2 + Q),0.01 (0.00-0.02),0.02 (0.00-0.04)
Kernel PCA (RBF),0.01 (0.00-0.03),0.02 (0.01-0.04)
"Recipe at this hour, stepped outputs (fixed after seeing results)",0.84 (0.83-0.84),0.84 (0.83-0.85)


In [10]:
round4.campaign_carry_counts()

detectors over 3%  detectors  \
setup  fault                                                    
recipe A: as published                           2         26   
       B: held at recipe values                  3         26   
       C: 80% of schedule                        3         26   
       C: 50% of schedule                        3         26   
       C: 20% of schedule                        5         26   
       aeration                                  4         26   
       back pressure                             0         26   
       base flow                                 6         26   
       coolant flow                              1         26   
mixed  A: as published                           2         26   
       A on operator batches                     2         26   

                                which (false alarms before; caught at the line -> at each batch's own line)  
setup  fault                                                                                                 
recipe A: as published           Isolation Forest (0.10; 0.01 -> 0.00); CVA (T2 + Q) (0.0...                 
       B: held at recipe values  Isolation Forest (0.09; 0.01 -> 0.00); Autoencoder, outp...                 
       C: 80% of schedule        Isolation Forest (0.09; 0.00 -> 0.00); Autoencoder, outp...                 
       C: 50% of schedule        Isolation Forest (0.11; 0.00 -> 0.00); Autoencoder, outp...                 
       C: 20% of schedule        Isolation Forest (0.11; 0.01 -> 0.00); Nearest normal re...                 
       aeration                  Isolation Forest (0.11; 0.12 -> 0.03); Curved-link, scor...                 
       back pressure                                                                                         
       base flow                 Rolling z-score (0.03; 0.53 -> 0.69); EWMA chart (0.03; ...                 
       coolant flow                                    Isolation Forest (0.04; 0.00 -> 0.00)                 
mixed  A: as published           Isolation Forest (0.10; 0.01 -> 0.00); CVA (T2 + Q) (0.0...                 
       A on operator batches     Isolation Forest (0.11; 0.01 -> 0.01); CVA (T2 + Q) (0.0...

**What this shows.**

- **No deep model beats the best simple detector on any generated fault**, as in round 3. On two faults the best deep model is now **behind** by the margin: the base flow fault, where the nearest normal reading on all sensors catches 0.95 against the LSTM's 0.69 (−26 points, range −30 to −19); and the sugar feed at 80% of schedule, where the recipe check on the stepped outputs catches every faulty reading against the autoencoder's 0.80 (−20). *The second rests on the fix made after seeing results;* without it that fault stays a tie (+3.7).
- **Recipe at this hour on the stepped outputs catches every faulty reading of faults A, B and C (all three sizes)**, and of the aeration and back pressure faults; it catches nothing of the base flow and coolant faults, whose outputs move continuously. In the mixed setup it catches 0.84 of fault A on recipe and on operator batches.
- **The unseen value check on all sensors** catches all of fault A in both setups and misses fault B entirely (by design, as the outputs-only version). **Dynamic PCA, CVA and kernel PCA** see almost nothing of the substrate faults (0.00 to 0.02) but catch the aeration and back pressure faults whole.
- **False alarms before the fault.** The isolation forest raises about 10% on every fault but catches almost nothing anyway. A few detectors sit just over 3% (CVA; the autoencoder on the outputs; on the base flow fault several detectors at 3%): at each batch's own line most of them catch about the same. The one that drops is **the curved-link scored like the autoencoder on the base flow fault: 0.92 at the line from the line batches, 0.72 at each batch's own line**. The round-3 figure for that fault was partly false alarms; the nearest normal reading's 0.95 stays 0.95.